# IDEAtlas — SDG 11.1.1 automation

This notebook runs the full mapping of a **city** (or of **your own custom
area**) for SDG indicator 11.1.1 — the proportion of urban population living
in slums. It:

1. connects to your **Google Drive** (all results are saved there),
2. downloads this project's code plus the pinned `ai-dua-mapping` framework,
3. builds the exact **Python 3.10 + TensorFlow** environment the project is
   tested with (Miniconda inside this Colab machine),
4. fetches the city boundary, downloads the satellite / building / GHSL
   layers, classifies the built-up area, and produces the SDG statistics
   plus a report.

**To run everything:** menu `Runtime` → **Run all**. You only edit the
*Settings* cell below. No Python knowledge needed.

Expected time: ≈10 min environment setup (first run only) + 20–40 min per
city. Prefer the **T4 GPU** runtime. Results land in
`MyDrive/IDEAtlas/<city>/`.

---

## Step 0 — Settings

Edit only the next cell (the grey *Settings* block). Everything else can
keep its defaults.

In [ ]:
# ====================================================================
#  SETTINGS — the only cell you need to edit
# ====================================================================

# GitHub URL of this project. The repository must be public, or use a
# personal-access-token URL (see notebooks/README.md).
REPO_URL = "https://github.com/bitsandbricks/IDEAtlas_automation"

# City to map: "encarnacion", "asuncion", "ciudad-del-este", or
# "custom" (then fill the CUSTOM_* lines below).
CITY = "encarnacion"

# Processing year (leave at 2025 unless the config/cities file differs).
YEAR = 2025

# ---- only used when CITY == "custom" ----
NEW_CITY    = "my-town"     # short lowercase name
NEW_COUNTRY = "paraguay"    # lowercase country, used in file names
# Full in-Drive path of a hand-made WGS84 boundary GeoJSON (see Step 5).
NEW_AOI_DRIVE = "/content/drive/MyDrive/my_aoi.geojson"

---

## Step 1 — Connect Google Drive

Run the next cell and click **Allow** when Google asks for permission.
Results will be written to `MyDrive/IDEAtlas/<city>/`.

In [ ]:
import os
from google.colab import drive

DRIVE_ROOT = "/content/drive/MyDrive"
if not os.path.isdir(DRIVE_ROOT):
    drive.mount("/content/drive")
print("Google Drive ready at:", DRIVE_ROOT)

---

## Step 2 — Check the GPU

The notebook prefers the free **T4 GPU** runtime. This step only reports
what is available; the same environment also runs on CPU (slower). Use
`Runtime` → `Change runtime type` → T4 GPU.

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout[:300])
else:
    print("No GPU detected — you can continue, but choosing the T4 GPU")
    print("runtime (Runtime -> Change runtime type) makes things much faster.")

---

## Step 3 — Get the code

Downloads this repository and the framework `ai-dua-mapping`, pinned to the
exact revision this project is tested against.

In [ ]:
import os, subprocess

REPO_ROOT = "/content/ideatlas"
os.makedirs(REPO_ROOT, exist_ok=True)

if "<your-github-user>" in REPO_URL:
    raise SystemExit("Set REPO_URL in the Settings cell to the real GitHub URL of this project.")

def run(cmd):
    print("$", cmd)
    return subprocess.run(["bash", "-lc", cmd], text=True, capture_output=True)

if not os.path.isdir(os.path.join(REPO_ROOT, ".git")):
    run(f"git clone '{REPO_URL}' '{REPO_ROOT}'")
run(f"git -C '{REPO_ROOT}' fetch --quiet origin")
run(f"git -C '{REPO_ROOT}' checkout --quiet main")          # branch of your public repo

FRAMEWORK_URL = "https://github.com/IDEAtlas/ai-dua-mapping.git"
FRAMEWORK_PIN = "b66926d5f54dd926815c787418a2ba8f2e62cf27"
if not os.path.isdir(os.path.join(REPO_ROOT, "ai-dua-mapping", ".git")):
    run(f"git clone '{FRAMEWORK_URL}' '{REPO_ROOT}/ai-dua-mapping'")
run(f"git -C '{REPO_ROOT}/ai-dua-mapping' checkout --quiet '{FRAMEWORK_PIN}'")

assert os.path.isfile(os.path.join(REPO_ROOT, "ai-dua-mapping", "environment.yaml")), \
    "framework clone failed"
commit = run(f"git -C '{REPO_ROOT}' log -1 --format='%h %s'").stdout.strip()

print("Automation repo ready at", REPO_ROOT)
print("  ->", commit)
print("Framework pinned at", FRAMEWORK_PIN[:7])

---

## Step 4 — Prepare the Python 3.10 environment

The project is tested with an exact set of package versions pinned to
**Python 3.10** in `ai-dua-mapping/environment.yaml`. Colab's own Python is
newer, so this step installs a **Miniconda** environment inside the machine
that matches the project exactly, and every later step runs inside it
(TensorFlow is configured for the GPU automatically).

First run installs ≈10 minutes; afterwards the step is skipped.

In [ ]:
import subprocess, sys

def run(cmd):
    return subprocess.run(["bash", "-lc", cmd], text=True, capture_output=True)

probe = run("command -v conda")
if probe.returncode == 0:
    print("Conda already installed:", probe.stdout.strip())
else:
    print("Installing Miniconda via condacolab (one-time) ...")
    print("Note: this install may restart the runtime once. If so, resume from Step 6.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "condacolab"], check=True)
    import condacolab
    condacolab.install()


## If Colab restarts during the install

Only the Python kernel restarts - your clone, Miniconda and environment are
safe. Do this:

1. Re-run the **next cell** (it finishes Step 4: activates `ideatlas` and
   checks TensorFlow).
2. Then continue from **Step 6** (the pipeline).

Only a full **Runtime > Factory reset runtime** wipes `/content` - re-run
the whole notebook in that case.


In [ ]:
import subprocess, os

def run(cmd, check=False):
    r = subprocess.run(["bash", "-lc", cmd], text=True, capture_output=True)
    if check and r.returncode:
        raise RuntimeError(r.stderr[-2000:])
    return r

run("apt-get install -y -q make >/dev/null 2>&1")   # Colab may lack make
base = run("conda info --base").stdout.strip()
if not base:
    raise SystemExit("conda is not on PATH - restart the runtime and Run all again.")
print("conda base:", base)
print("Step 4 - finishing environment setup ...")

existing = run(f"'{base}/bin/conda' env list").stdout
if "ideatlas" in existing:
    print("Environment 'ideatlas' already exists - skipping creation.")
else:
    print("Creating 'ideatlas' environment from ai-dua-mapping/environment.yaml (~10 min) ...")
    run(f"'{base}/bin/conda' env create -f /content/ideatlas/ai-dua-mapping/environment.yaml", check=True)

# Activate without sourcing conda.sh: put the env binaries on PATH and,
# like the Makefile, expose the env lib dir so TensorFlow finds cudatoolkit/cudnn.
env_bin = f"{base}/envs/ideatlas/bin"
env_lib = f"{base}/envs/ideatlas/lib"
prefix = f"export PATH=\"{env_bin}:$PATH\"; export LD_LIBRARY_PATH=\"{env_lib}:$LD_LIBRARY_PATH\"; "
run(prefix + "pip install -q pyyaml openpyxl requests", check=True)
_check = (
    "import tensorflow, sys; "
    "print('env python', sys.version); "
    "print('TensorFlow', tensorflow.__version__, 'GPU devices:', "
    "tensorflow.config.list_physical_devices('GPU'))\n"
)
with open("/content/check_env.py", "w") as fh:
    fh.write(_check)
run(prefix + "python /content/check_env.py", check=True)


---

## Step 5 — Custom city (only when `CITY = "custom"`)

If you chose one of the **example cities**, skip to Step 6.

For a **custom** area you provide a boundary GeoJSON (WGS84, a single
`Polygon`/`MultiPolygon` in a FeatureCollection — see `docs/adding-a-city.md`,
Option B). Upload the file to Google Drive, then put its full path in
`NEW_AOI_DRIVE` in the Settings cell.

In [ ]:
import os, shutil

if CITY == "custom":
    if not (NEW_CITY and NEW_COUNTRY and NEW_AOI_DRIVE):
        raise SystemExit("CITY='custom' requires NEW_CITY, NEW_COUNTRY and NEW_AOI_DRIVE.")
    if not os.path.isfile(NEW_AOI_DRIVE):
        raise SystemExit(f"AOI file not found: {NEW_AOI_DRIVE}")
    norm = NEW_CITY.strip().lower().replace(" ", "_").replace("-", "_")
    cnorm = NEW_COUNTRY.strip().lower().replace(" ", "_").replace("-", "_")
    aoi_dir = os.path.join(REPO_ROOT, "ai-dua-mapping", "data", "raw", "aoi")
    os.makedirs(aoi_dir, exist_ok=True)
    # The framework reads data/raw/aoi/<city>_<country>_aoi.geojson
    # (pipeline/config.py: city_normalized = "<city>_<country>").
    dst = os.path.join(aoi_dir, f"{norm}_{cnorm}_aoi.geojson")
    shutil.copy(NEW_AOI_DRIVE, dst)
    cfg_dir = os.path.join(REPO_ROOT, "config", "cities")
    os.makedirs(cfg_dir, exist_ok=True)
    with open(os.path.join(cfg_dir, f"{NEW_CITY}.yaml"), "w") as fh:
        fh.write(f"# {NEW_CITY} ({NEW_COUNTRY}).\ncountry: {NEW_COUNTRY}\nyear: {YEAR}\n")
    print(f"Custom city '{NEW_CITY}' prepared.")
    print("  AOI ->", dst)
else:
    print("Using example city:", CITY)

---

## Step 6 — Run the pipeline

Fetches the city boundary, downloads all remote layers (Sentinel-2, Google
Open Buildings, GHSL), classifies the built-up area and writes the
SDG 11.1.1 statistics. Takes roughly **20–40 minutes** per city. Progress is
streamed below; ignore the noisy TensorFlow messages.

In [ ]:
import subprocess, os, sys

city = CITY if CITY != "custom" else NEW_CITY

# Same activation as the environment cell: env bin on PATH (and the env
# lib dir, which the Makefile also uses for LD_LIBRARY_PATH).
base = subprocess.run(["bash", "-lc", "conda info --base"], text=True, capture_output=True).stdout.strip()
env_bin = f"{base}/envs/ideatlas/bin"
env_lib = f"{base}/envs/ideatlas/lib"
prefix = (f"export PATH=\"{env_bin}:$PATH\"; "
          f"export LD_LIBRARY_PATH=\"{env_lib}:$LD_LIBRARY_PATH\"; "
          "cd /content/ideatlas;")

def run(cmd, check=True):
    print("\n$", cmd, flush=True)
    r = subprocess.run(["bash", "-lc", cmd], text=True, capture_output=True)
    if r.stdout:
        print(r.stdout, end="" if r.stdout.endswith("\n") else "\n")
    if check and r.returncode:
        print("--- stderr (tail) ---", file=sys.stderr)
        print(r.stderr[-3000:], file=sys.stderr)
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")

if CITY != "custom":
    run(prefix + " make aois")

run(prefix + f" make city CITY='{city}' YEAR={YEAR} TASK=classify")
run(prefix + " make report")


---

## Step 7 — Copy the results to Google Drive

Copies both a lightweight **summary** (statistics + reports) and the **full
outputs** (maps, masks, processed layers, logs) into
`MyDrive/IDEAtlas/<city>/`. The full part can be hundreds of MB.

In [ ]:
import os, shutil, glob, json

norm = (CITY if CITY != "custom" else NEW_CITY).strip().lower().replace(" ", "_").replace("-", "_")
city_key = CITY if CITY != "custom" else NEW_CITY
dest = os.path.join(DRIVE_ROOT, "IDEAtlas", norm)
sum_dir = os.path.join(dest, "SUMMARY")
full_dir = os.path.join(dest, "FULL")
os.makedirs(sum_dir, exist_ok=True)
os.makedirs(full_dir, exist_ok=True)

ROOT = REPO_ROOT
def copy(src, dst_dir):
    if os.path.isfile(src):
        shutil.copy(src, os.path.join(dst_dir, os.path.basename(src))); return [src]
    if os.path.isdir(src):
        out = os.path.join(dst_dir, os.path.basename(src)); shutil.copytree(src, out, dirs_exist_ok=True); return [src]
    return []

moved = []
# Summary
for f in glob.glob(os.path.join(ROOT, "outputs", "*_sdg_stats.json")):
    moved += copy(f, sum_dir)
for f in glob.glob(os.path.join(ROOT, "outputs", "ideatlas_report.*")):
    moved += copy(f, sum_dir)
# Full outputs
for f in glob.glob(os.path.join(ROOT, "ai-dua-mapping", "output", f"{norm}*.tif")):
    moved += copy(f, full_dir)
for d in glob.glob(os.path.join(ROOT, "ai-dua-mapping", "data", "processed", f"{norm}*")):
    moved += copy(d, full_dir)
for d in glob.glob(os.path.join(ROOT, "data", "processed", f"{city_key}*")):
    moved += copy(d, full_dir)
for f in glob.glob(os.path.join(ROOT, "ai-dua-mapping", "data", "raw", "aoi", f"{norm}*.geojson")):
    moved += copy(f, full_dir)
for f in glob.glob(os.path.join(ROOT, "ai-dua-mapping", "log", "*.log")):
    moved += copy(f, full_dir)

print(f"Copied {len(moved)} items into {dest}")
for s in sorted(set(moved)):
    print("  ", os.path.relpath(s, ROOT))
for f in glob.glob(os.path.join(sum_dir, "*_sdg_stats.json")):
    try:
        stats = json.load(open(f))
    except Exception:
        continue
    print("\nSDG stats:")
    for k, v in stats.items():
        print(f"  {k}: {v}")

---

## Done

Your results are in **`MyDrive/IDEAtlas/<city>/`** (Google Drive):

* `SUMMARY/` — SDG 11.1.1 statistics (`.json`) and the human-readable report
  (`.md` + `.xlsx`),
* `FULL/` — everything the pipeline produced (maps, masks, processed layers,
  logs, AOI).

To map a different city: change `CITY` in the Settings cell and **Run all**
again (every step is skip-safe if already done). For your own area instead
of a city, use `CITY = "custom"` and provide a boundary file (Step 5).

See `notebooks/README.md` for tips and the full documentation.